In [2]:
import cv2
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import os

import tempfile

from anomalib.models import Padim
from anomalib.engine import Engine
from anomalib.data import Folder
from anomalib.data import PredictDataset

from lightning.pytorch.callbacks import ModelCheckpoint

W0724 16:43:19.100000 24424 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
d:\Projects\VisionXM\.visionXM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projects\VisionXM\.visionXM\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [6]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

print(DATA_DIR)
print(DATA_DIR.exists())

d:\Projects\VisionXM\data
True


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [8]:
datamodule = Folder(
    name="screws",
    root=str(DATA_DIR),
    normal_dir="train/good",
    abnormal_dir=[
        "test/manipulated_front",
        "test/scratch_head",
        "test/scratch_neck",
        "test/thread_side",
        "test/thread_top",
    ],
    normal_test_dir="test/good",
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=4,
)

In [3]:
model = Padim(backbone="resnet18", layers=["layer1","layer2", "layer3"])

In [11]:
engine = Engine(accelerator="gpu")

engine.fit(model=model, datamodule=datamodule)

# Save the trained model
engine.trainer.save_checkpoint(".\checkpoints\padim_checkpoint.ckpt")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor  │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator     │      0 │ train │     0 │
│ 3 │ model          │ PadimModel    │  2.8 M │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.131                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


`weights_only` was not set, defaulting to `False`.


In [21]:
engine.test(model=model, datamodule=datamodule)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


d:\Projects\VisionXM\.visionXM\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Projects\VisionXM\.visionXM\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: The ``compute`` 
method of metric AUROC was called before the ``update`` method which may lead to errors, as metric states have not 
yet been updated.
  warnings.warn(*args, **kwargs)

d:\Projects\VisionXM\.visionXM\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: The ``compute`` 
method of metric F1Score was called before the ``update`` method which may lead to errors, as metric states have 
not yet been updated.
  warnings.warn(*args, **kwargs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.8238095045089722     │
│       image_F1Score       │    0.8507462739944458     │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 0.8238095045089722, 'image_F1Score': 0.8507462739944458}]

In [22]:
predictions = engine.predict(model=model, datamodule=datamodule, ckpt_path=".\checkpoints\padim_checkpoint.ckpt")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at D:\Projects\VisionXM\notebooks\checkpoints\padim_checkpoint.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at D:\Projects\VisionXM\notebooks\checkpoints\padim_checkpoint.ckpt
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.


In [14]:
def predict_crop(crop):

    temp_file = "D:\Projects\VisionXM\data\test\thread_side\006.png"
    cv2.imwrite(temp_file, crop)

    dataset = PredictDataset(path=temp_file)

    predictions = engine.predict(
        model=model,
        dataset=dataset,
        ckpt_path=str("checkpoints\padim_checkpoint.ckpt"),
    )

    result = predictions[0]

    score = float(result.pred_score)

    label = "Defective" if result.pred_label else "Good"

    color = (0,0,255) if label=="Defective" else (0,255,0)

    os.remove(temp_file)

    return label, score, color

In [16]:
img = cv2.imread(Path(r"D:\Projects\VisionXM\data\train\good\007.png"))

label, score, color = predict_crop(img)

print(label)
print(score)

ValueError: Path contains non-printable characters: D:\Projects\VisionXM\data	est	hread_side.png